In [1]:
#!pip install -qqq arize-otel

In [2]:
import os
space_id = os.environ["ARIZE_SPACE_ID"]
api_key = os.environ["ARIZE_API_KEY"]

In [3]:
project_name = "streamflix_search_augment"

In [4]:
# Import open-telemetry dependencies
from arize.otel import register
import pandas as pd
import json

tracer_provider = register(
    space_id=space_id,
    api_key=api_key,
    project_name=project_name,
)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: streamflix_search_augment
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
# Bedrock metadata (for span attributes only; no real API calls)
BEDROCK_REGION = "us-east-1"
BEDROCK_MODEL_ID = "claude-3-7-sonnet-latest"

In [6]:
# StreamFlix Search Augment Data Configuration
# User queries, catalog (id, title, genre, description), first-pull, reranked list, personalized response

import random

# Catalog: synthetic titles for retrieval/rerank
CATALOG = [
    {"id": "t1", "title": "Comedy Special: Late Night Laughs", "genre": "Comedy", "description": "Stand-up special with topical humor."},
    {"id": "t2", "title": "Friday Feel-Good", "genre": "Comedy", "description": "Light romantic comedy set in a small town."},
    {"id": "t3", "title": "The Twist", "genre": "Thriller", "description": "Psychological thriller with an unexpected ending."},
    {"id": "t4", "title": "Shadow Protocol", "genre": "Thriller", "description": "Spy thriller with multiple plot twists."},
    {"id": "t5", "title": "Cosmos Revisited", "genre": "Documentary", "description": "Space exploration and the universe."},
    {"id": "t6", "title": "Beyond Earth", "genre": "Documentary", "description": "Documentary about Mars missions and future colonization."},
    {"id": "t7", "title": "Weekend Binge", "genre": "Comedy", "description": "Sitcom about friends on a weekend trip."},
    {"id": "t8", "title": "Dark Signal", "genre": "Thriller", "description": "Mystery thriller with a twist in the final act."},
]

# Per-query scenario: query text, first_pull (catalog ids + scores), reranked (reordered ids + scores), response
SEARCH_AUGMENT_CONFIG = [
    {
        "query": "Something light and funny for Friday night",
        "first_pull": [("t1", 0.82), ("t2", 0.79), ("t7", 0.76), ("t3", 0.52), ("t4", 0.48)],
        "reranked": [("t2", 0.94), ("t7", 0.91), ("t1", 0.88)],
        "response": "For Friday night I'd go with **Friday Feel-Good** or **Weekend Binge**—both are light and easy to watch. If you want laughs, **Late Night Laughs** is a solid pick.",
    },
    {
        "query": "thriller with a twist",
        "first_pull": [("t3", 0.85), ("t4", 0.81), ("t8", 0.78), ("t1", 0.45), ("t2", 0.41)],
        "reranked": [("t3", 0.96), ("t8", 0.93), ("t4", 0.89)],
        "response": "You’ll get a real twist in **The Twist** and **Dark Signal**—both deliver that payoff. **Shadow Protocol** is another strong option if you like spy thrillers.",
    },
    {
        "query": "documentary about space",
        "first_pull": [("t5", 0.88), ("t6", 0.84), ("t3", 0.44), ("t1", 0.40)],
        "reranked": [("t5", 0.95), ("t6", 0.92)],
        "response": "For space docs, **Cosmos Revisited** and **Beyond Earth** are perfect—both cover exploration and the universe in a really engaging way.",
    },
    {
        "query": "something I can binge this weekend",
        "first_pull": [("t7", 0.83), ("t1", 0.77), ("t2", 0.74), ("t5", 0.55)],
        "reranked": [("t7", 0.93), ("t1", 0.90), ("t2", 0.87)],
        "response": "**Weekend Binge** is literally about a weekend trip with friends—great for binging. **Late Night Laughs** and **Friday Feel-Good** also work well back-to-back.",
    },
    {
        "query": "scary thriller with a good ending",
        "first_pull": [("t8", 0.86), ("t3", 0.83), ("t4", 0.80), ("t6", 0.48)],
        "reranked": [("t8", 0.95), ("t3", 0.92), ("t4", 0.88)],
        "response": "**Dark Signal** and **The Twist** both have strong, satisfying endings. **Shadow Protocol** is another good choice if you like spy thrillers with a twist.",
    },
    {
        "query": "funny show to relax",
        "first_pull": [("t1", 0.84), ("t2", 0.81), ("t7", 0.79), ("t5", 0.47)],
        "reranked": [("t1", 0.92), ("t7", 0.90), ("t2", 0.87)],
        "response": "To relax and laugh, try **Late Night Laughs** or **Weekend Binge**. **Friday Feel-Good** is a nice light option too.",
    },
]

def _catalog_by_id():
    return {item["id"]: item for item in CATALOG}

def get_search_augment_data(query_id=None):
    """Get data for one search-augment trace. query_id: 0..len(SEARCH_AUGMENT_CONFIG)-1 or None for random."""
    configs = SEARCH_AUGMENT_CONFIG
    if query_id is None:
        query_id = random.randint(0, len(configs) - 1)
    idx = query_id % len(configs)
    cfg = configs[idx]
    catalog_map = _catalog_by_id()
    # First-pull as retrieved documents (id, content, score)
    retrieved = []
    for doc_id, score in cfg["first_pull"]:
        item = catalog_map.get(doc_id, {"id": doc_id, "title": doc_id, "genre": "", "description": ""})
        content = f"{item['title']} ({item['genre']}). {item['description']}"
        retrieved.append({"id": doc_id, "content": content, "score": round(score + random.uniform(-0.02, 0.02), 2)})
    # Reranked: same structure, reordered
    reranked = []
    for doc_id, score in cfg["reranked"]:
        item = catalog_map.get(doc_id, {"id": doc_id, "title": doc_id, "genre": "", "description": ""})
        content = f"{item['title']} ({item['genre']}). {item['description']}"
        reranked.append({"id": doc_id, "content": content, "score": round(score + random.uniform(-0.01, 0.01), 2)})
    return {
        "query_id": idx,
        "query": cfg["query"],
        "retrieved_docs": retrieved,
        "reranked_docs": reranked,
        "response": cfg["response"],
    }

In [7]:
# Prompt template for LLM: personalize_recommendations (query + reranked list -> conversational response)
PROMPT_TEMPLATE_CONFIG = {
    "personalize_recommendations": {
        "template": (
            "The user asked: \"{query}\"\n\n"
            "Reranked recommendations (use only these):\n{reranked_list}\n\n"
            "Produce a short conversational response recommending 2-3 titles from the list. Be natural and concise."
        ),
        "system_message": "You are a StreamFlix search assistant. Given the user's query and a reranked list of titles, reply with a brief, friendly recommendation of 2-3 titles. Use only titles from the list.",
        "version": "v1.0",
    }
}

In [8]:
# Timer utility
from contextlib import contextmanager
import time

@contextmanager
def timer():
    start = time.time()
    yield
    print(f"Execution time: {time.time() - start:.2f} seconds")

In [9]:
# Trace: CHAIN -> RETRIEVER (candidate_retrieval) -> RERANKER (rerank_candidates) -> AGENT -> LLM (personalize_recommendations)
from opentelemetry import trace
from opentelemetry.trace.status import Status, StatusCode
import json
import time
import random

def create_search_augment_trace(tracer, query_id=None, session_id=None):
    """Create one StreamFlix search-augment trace. Returns dict with span_data."""
    data = get_search_augment_data(query_id)
    query = data["query"]
    retrieved_docs = data["retrieved_docs"]
    reranked_docs = data["reranked_docs"]
    response = data["response"]

    cfg = PROMPT_TEMPLATE_CONFIG.get("personalize_recommendations", {})
    reranked_list = "\n".join(f"- {d['id']}: {d['content'][:80]}..." for d in reranked_docs)
    user_msg = cfg.get("template", "").format(query=query, reranked_list=reranked_list)
    system_msg = cfg.get("system_message", "You are a helpful assistant.")

    chain_name = "StreamFlix Search Augment Pipeline"
    retrieval_latency_ms = random.uniform(50, 200)
    reranker_latency_ms = random.uniform(80, 250)
    agent_latency_ms = random.uniform(30, 120)
    llm_latency_ms = random.uniform(800, 2500)
    chain_latency_ms = retrieval_latency_ms + reranker_latency_ms + agent_latency_ms + llm_latency_ms + random.uniform(20, 80)

    chain_span_id = retrieval_span_id = reranker_span_id = agent_span_id = llm_span_id = None

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", query)
        chain_span.set_attribute("input.mime_type", "text/plain")
        if session_id:
            chain_span.set_attribute("session.id", session_id)
        time.sleep(0.003)
        ctx = chain_span.get_span_context()
        chain_span_id = format(ctx.span_id, "016x")

        with tracer.start_as_current_span("candidate_retrieval") as ret_span:
            ret_span.set_attribute("openinference.span.kind", "RETRIEVER")
            ret_span.set_attribute("input.value", json.dumps({"query": query}))
            ret_span.set_attribute("input.mime_type", "application/json")
            for idx, doc in enumerate(retrieved_docs):
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.id", doc["id"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
            ret_span.set_attribute("output.value", json.dumps([{"id": d["id"], "score": d["score"]} for d in retrieved_docs]))
            ret_span.set_attribute("output.mime_type", "application/json")
            ret_span.set_attribute("latency_ms", round(retrieval_latency_ms, 2))
            ret_span.set_status(Status(StatusCode.OK))
            ctx = ret_span.get_span_context()
            retrieval_span_id = format(ctx.span_id, "016x")

        with tracer.start_as_current_span("rerank_candidates") as rer_span:
            rer_span.set_attribute("openinference.span.kind", "RERANKER")
            rer_span.set_attribute("input.value", json.dumps({"query": query, "document_ids": [d["id"] for d in retrieved_docs], "scores": [d["score"] for d in retrieved_docs]}))
            rer_span.set_attribute("output.value", json.dumps([{"id": d["id"], "score": d["score"]} for d in reranked_docs]))
            rer_span.set_attribute("reranker.model_name", "cross-encoder-reranker-v1")
            rer_span.set_attribute("latency_ms", round(reranker_latency_ms, 2))
            rer_span.set_status(Status(StatusCode.OK))
            ctx = rer_span.get_span_context()
            reranker_span_id = format(ctx.span_id, "016x")

        with tracer.start_as_current_span("search_augment_agent") as agent_span:
            agent_span.set_attribute("openinference.span.kind", "AGENT")
            agent_span.set_attribute("agent.name", "search_augment_agent")
            agent_input = json.dumps({"request": "personalized_recommendations", "query": query, "instruction": "Retrieve -> Rerank -> Personalize with LLM."})
            agent_span.set_attribute("input.value", agent_input)
            agent_span.set_attribute("input.mime_type", "application/json")
            if session_id:
                agent_span.set_attribute("session.id", session_id)
            agent_span.set_attribute("latency_ms", round(agent_latency_ms, 2))
            time.sleep(0.002)
            ctx = agent_span.get_span_context()
            agent_span_id = format(ctx.span_id, "016x")

            with tracer.start_as_current_span("personalize_recommendations") as llm_span:
                if session_id:
                    llm_span.set_attribute("session.id", session_id)
                llm_span.set_attribute("openinference.span.kind", "LLM")
                llm_span.set_attribute("llm.model_name", BEDROCK_MODEL_ID)
                llm_span.set_attribute("llm.provider", "aws")
                llm_span.set_attribute("llm.system", "anthropic")
                llm_span.set_attribute("llm.input_messages.0.message.role", "system")
                llm_span.set_attribute("llm.input_messages.0.message.content", system_msg)
                llm_span.set_attribute("llm.input_messages.1.message.role", "user")
                llm_span.set_attribute("llm.input_messages.1.message.content", user_msg[:2000])
                llm_span.set_attribute("llm.output_messages.0.message.role", "assistant")
                llm_span.set_attribute("llm.output_messages.0.message.content", response)
                llm_span.set_attribute("llm.token_count.prompt", 400)
                llm_span.set_attribute("llm.token_count.completion", 120)
                llm_span.set_attribute("llm.token_count.total", 520)
                llm_span.set_attribute("input.value", user_msg[:2000])
                llm_span.set_attribute("output.value", response)
                llm_span.set_attribute("latency_ms", round(llm_latency_ms, 2))
                llm_span.set_status(Status(StatusCode.OK))
                ctx = llm_span.get_span_context()
                llm_span_id = format(ctx.span_id, "016x")

            agent_span.set_attribute("output.value", response)
            agent_span.set_attribute("output.mime_type", "text/plain")

        chain_span.set_attribute("output.value", response[:500])
        chain_span.set_attribute("output.mime_type", "text/plain")
        chain_span.set_attribute("latency_ms", round(chain_latency_ms, 2))
        chain_span.set_status(Status(StatusCode.OK))

    return {
        "span_data": {
            "chain_span_id": chain_span_id,
            "retrieval_span_id": retrieval_span_id,
            "reranker_span_id": reranker_span_id,
            "agent_span_id": agent_span_id,
            "llm_span_id": llm_span_id,
            "query": query,
            "response": response,
            "query_id": data["query_id"],
            "session_id": session_id,
        }
    }

In [10]:
# Session evaluations from CHAIN spans (grouped by session_id)
def generate_session_evaluations_from_chain_spans(span_df: pd.DataFrame) -> list:
    session_evaluations = []
    chain_spans = span_df[span_df["openinference.span.kind"] == "CHAIN"].copy()
    if len(chain_spans) == 0:
        return session_evaluations
    if "session_id" not in chain_spans.columns:
        return session_evaluations
    chain_spans = chain_spans[chain_spans["session_id"].notna()]
    if len(chain_spans) == 0:
        return session_evaluations
    if "timestamp" not in chain_spans.columns:
        chain_spans["timestamp"] = chain_spans.index
    chain_spans_sorted = chain_spans.sort_values("timestamp")
    oldest_per_session = chain_spans_sorted.groupby("session_id").first().reset_index()
    print(f"  Processing {len(oldest_per_session):,} unique sessions for session evaluations")
    for _, row in oldest_per_session.iterrows():
        span_id = row.get("context.span_id", "")
        session_id = row.get("session_id", "unknown")
        will_pass = random.random() < 0.82
        label = "pass" if will_pass else "fail"
        score = 0 if will_pass else 1
        explanations = [
            "Search augment pipeline passed: Retrieval, rerank, and personalization worked together.",
            "Session validation successful: The search augment pipeline executed correctly.",
        ] if will_pass else [
            "Search augment pipeline failed: Coordination or execution issues.",
            "Session validation failed: The chain did not orchestrate retrieval, rerank, and LLM effectively.",
        ]
        session_evaluations.append({
            "context.span_id": span_id,
            "session_eval.SearchAugmentSession.label": label,
            "session_eval.SearchAugmentSession.score": score,
            "session_eval.SearchAugmentSession.explanation": random.choice(explanations),
        })
    return session_evaluations

In [11]:
# Batch collection: CHAIN, RETRIEVER, RERANKER, AGENT, LLM for evaluation generation
print("Batch cell collects CHAIN, RETRIEVER, RERANKER, AGENT, LLM for evaluation generation.")

Batch cell collects CHAIN, RETRIEVER, RERANKER, AGENT, LLM for evaluation generation.


In [12]:
# Evaluation generation: AGENT, LLM (relevance + hallucination)
def generate_agent_trajectory_accuracy_eval(span_data: dict) -> dict:
    will_pass = random.random() < 0.75
    if will_pass:
        label, score = "pass", 0
        explanations = ["Search augment orchestration passed: Agent correctly sequenced retrieval, rerank, and LLM personalization.",]
    else:
        label, score = "fail", 1
        explanations = ["Search augment orchestration failed: Agent deviated from expected sequence.",]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_hallucination_eval(span_data: dict) -> dict:
    will_pass = random.random() < 0.85
    if will_pass:
        label, score = "pass", 0
        explanations = ["Hallucination check passed: Response is consistent with reranked recommendations.",]
    else:
        label, score = "fail", 1
        explanations = ["Hallucination detected: Response contains titles or details not in the list.",]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_relevance_eval(span_data: dict) -> dict:
    will_pass = random.random() < 0.88
    if will_pass:
        label, score = "pass", 0
        explanations = ["Relevance passed: Recommendations match the user query intent.",]
    else:
        label, score = "fail", 1
        explanations = ["Relevance failed: Recommendations do not adequately address the query.",]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def create_evaluations_from_span_data(span_df: pd.DataFrame) -> pd.DataFrame:
    """Create evaluation DataFrame. Handles CHAIN (skip), RETRIEVER (skip), RERANKER (skip), AGENT, LLM."""
    evaluations = []
    for _, row in span_df.iterrows():
        span_kind = row.get("openinference.span.kind", "")
        span_id = row.get("context.span_id", "")
        span_data = {"query": row.get("query", ""), "response": row.get("response", "")}
        if span_kind == "AGENT":
            eval_data = generate_agent_trajectory_accuracy_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "trace_eval.AgentTrajectoryAccuracy.label": eval_data["label"],
                "trace_eval.AgentTrajectoryAccuracy.score": eval_data["score"],
                "trace_eval.AgentTrajectoryAccuracy.explanation": eval_data["explanation"],
            })
        elif span_kind == "LLM":
            eval_data = generate_hallucination_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "eval.Hallucination.label": eval_data["label"],
                "eval.Hallucination.score": eval_data["score"],
                "eval.Hallucination.explanation": eval_data["explanation"],
            })
            rel_data = generate_relevance_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "eval.Relevance.label": rel_data["label"],
                "eval.Relevance.score": rel_data["score"],
                "eval.Relevance.explanation": rel_data["explanation"],
            })
    return pd.DataFrame(evaluations)

# Single Search Augment Trace Test

In [13]:
# Test one trace
tracer = trace.get_tracer(__name__)
with timer():
    result = create_search_augment_trace(tracer=tracer, query_id=0, session_id="test_session_001")
    sd = result["span_data"]
    print(f"Created search augment trace for query: {sd['query'][:50]}...")
    print(f"  chain_span_id: {sd['chain_span_id']}, agent_span_id: {sd['agent_span_id']}, llm_span_id: {sd['llm_span_id']}")
    print(f"  retrieval_span_id: {sd['retrieval_span_id']}, reranker_span_id: {sd['reranker_span_id']}")
    print("Created test trace – check in Arize for data.")
    flush_success = trace.get_tracer_provider().force_flush(timeout_millis=30000)
    if flush_success:
        print("Single trace exported successfully")
    else:
        print("Flush timeout - span may not have been exported")

Created search augment trace for query: Something light and funny for Friday night...
  chain_span_id: d61efc3d8b9ae7b8, agent_span_id: dba748893d3936f6, llm_span_id: e19923c634e7d7d1
  retrieval_span_id: f779f354692fd81d, reranker_span_id: f173d14eada6579e
Created test trace – check in Arize for data.
Single trace exported successfully
Execution time: 0.27 seconds


# Full batch search augment trace creation

In [14]:
# Batch generation: create_search_augment_trace for each run; collect CHAIN, RETRIEVER, RERANKER, AGENT, LLM
NUM_SPANS = 500
QUERY_IDS = list(range(len(SEARCH_AUGMENT_CONFIG)))
tracer = trace.get_tracer(__name__)
print(f"Generating {NUM_SPANS:,} search augment traces (CHAIN -> RETRIEVER -> RERANKER -> AGENT -> LLM)...")
print("Note: BatchSpanProcessor batches automatically.\n")

with timer():
    span_data_list = []
    SESSION_SIZE_MIN = 2
    SESSION_SIZE_MAX = 5
    current_session_id = None
    traces_in_current_session = 0
    session_size = 0
    BATCH_SIZE = 50

    for batch_start in range(0, NUM_SPANS, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, NUM_SPANS)
        for i in range(batch_start, batch_end):
            if current_session_id is None or traces_in_current_session >= session_size:
                current_session_id = f"session_{random.randint(100000, 999999)}"
                session_size = random.randint(SESSION_SIZE_MIN, SESSION_SIZE_MAX)
                traces_in_current_session = 0
            query_id = random.choice(QUERY_IDS)
            result = create_search_augment_trace(tracer=tracer, query_id=query_id, session_id=current_session_id)
            traces_in_current_session += 1

            if "span_data" in result:
                sd = result["span_data"]
                span_data_list.append({"context.span_id": sd["chain_span_id"], "openinference.span.kind": "CHAIN", "query": sd["query"], "response": sd["response"], "session_id": sd.get("session_id")})
                span_data_list.append({"context.span_id": sd["retrieval_span_id"], "openinference.span.kind": "RETRIEVER", "query": sd["query"], "response": sd["response"]})
                span_data_list.append({"context.span_id": sd["reranker_span_id"], "openinference.span.kind": "RERANKER", "query": sd["query"], "response": sd["response"]})
                span_data_list.append({"context.span_id": sd["agent_span_id"], "openinference.span.kind": "AGENT", "query": sd["query"], "response": sd["response"]})
                span_data_list.append({"context.span_id": sd["llm_span_id"], "openinference.span.kind": "LLM", "query": sd["query"], "response": sd["response"]})

        if (batch_end % 500 == 0) or (batch_end == NUM_SPANS):
            print(f"  Created {batch_end:,} / {NUM_SPANS:,} traces...")

    print(f"\nCompleted creating {NUM_SPANS:,} traces. Flushing remaining spans...")
    flush_timeout = min(30000 + (NUM_SPANS // 10000) * 10000, 300000)
    flush_success = trace.get_tracer_provider().force_flush(timeout_millis=flush_timeout)
    if flush_success:
        print(f"Flush successful. Collected {len(span_data_list):,} span records for evaluation generation.")
    else:
        print("Flush timeout – some spans may not have been exported.")

collected_span_data = span_data_list

Generating 500 search augment traces (CHAIN -> RETRIEVER -> RERANKER -> AGENT -> LLM)...
Note: BatchSpanProcessor batches automatically.

  Created 500 / 500 traces...

Completed creating 500 traces. Flushing remaining spans...
Flush successful. Collected 2,500 span records for evaluation generation.
Execution time: 3.70 seconds


In [ ]:
# Evaluation Generation and Logging - run AFTER batch span generation
print("=" * 60)
print("Generating evaluations from collected span data...")
print("=" * 60 + "\n")

if "collected_span_data" not in globals() or not collected_span_data:
    print("No span data found. Please run the span generation cell first.")
else:
    span_df = pd.DataFrame(collected_span_data)
    print(f"Collected {len(span_df):,} spans")
    print(f"  - CHAIN: {len(span_df[span_df['openinference.span.kind'] == 'CHAIN']):,}")
    print(f"  - RETRIEVER: {len(span_df[span_df['openinference.span.kind'] == 'RETRIEVER']):,}")
    print(f"  - RERANKER: {len(span_df[span_df['openinference.span.kind'] == 'RERANKER']):,}")
    print(f"  - AGENT: {len(span_df[span_df['openinference.span.kind'] == 'AGENT']):,}")
    print(f"  - LLM: {len(span_df[span_df['openinference.span.kind'] == 'LLM']):,}")
    print("\nGenerating evaluations for AGENT, LLM spans...")
    eval_df = create_evaluations_from_span_data(span_df)
    print("\nGenerating session evaluations from CHAIN spans...")
    session_evaluations = generate_session_evaluations_from_chain_spans(span_df)
    if session_evaluations:
        session_eval_df = pd.DataFrame(session_evaluations)
        eval_df = pd.concat([eval_df, session_eval_df], ignore_index=True)
        print(f"  Added {len(session_evaluations):,} session evaluations")
    if len(eval_df) > 0:
        print(f"\nGenerated {len(eval_df):,} evaluations\n")
        agent_evals = len(eval_df[eval_df["trace_eval.AgentTrajectoryAccuracy.label"].notna()]) if "trace_eval.AgentTrajectoryAccuracy.label" in eval_df.columns else 0
        hall_evals = len(eval_df[eval_df["eval.Hallucination.label"].notna()]) if "eval.Hallucination.label" in eval_df.columns else 0
        rel_evals = len(eval_df[eval_df["eval.Relevance.label"].notna()]) if "eval.Relevance.label" in eval_df.columns else 0
        session_evals = len(eval_df[eval_df["session_eval.SearchAugmentSession.label"].notna()]) if "session_eval.SearchAugmentSession.label" in eval_df.columns else 0
        print(f"  - AgentTrajectoryAccuracy: {agent_evals:,}")
        print(f"  - Hallucination: {hall_evals:,}")
        print(f"  - Relevance: {rel_evals:,}")
        print(f"  - Session evaluations: {session_evals:,}")
        if "trace_eval.AgentTrajectoryAccuracy.label" in eval_df.columns:
            print(f"    AgentTrajectoryAccuracy: {(eval_df['trace_eval.AgentTrajectoryAccuracy.label'] == 'pass').sum():,} pass, {(eval_df['trace_eval.AgentTrajectoryAccuracy.label'] == 'fail').sum():,} fail")
        if "eval.Hallucination.label" in eval_df.columns:
            print(f"    Hallucination: {(eval_df['eval.Hallucination.label'] == 'pass').sum():,} pass, {(eval_df['eval.Hallucination.label'] == 'fail').sum():,} fail")
        if "eval.Relevance.label" in eval_df.columns:
            print(f"    Relevance: {(eval_df['eval.Relevance.label'] == 'pass').sum():,} pass, {(eval_df['eval.Relevance.label'] == 'fail').sum():,} fail")
        if "session_eval.SearchAugmentSession.label" in eval_df.columns:
            print(f"    Session evaluation: {(eval_df['session_eval.SearchAugmentSession.label'] == 'pass').sum():,} pass, {(eval_df['session_eval.SearchAugmentSession.label'] == 'fail').sum():,} fail")
        print("\n" + "=" * 60)
        print("Logging evaluations to Arize...")
        print("=" * 60 + "\n")
        try:
            from arize.pandas.logger import Client
            client = Client(space_id=space_id, api_key=api_key)
            chunk_size = 500
            total_logged = 0
            # Log by eval type: only send rows that have that eval (no n/a placeholders)
            agent_col = "trace_eval.AgentTrajectory.label" if "trace_eval.AgentTrajectory.label" in eval_df.columns else "trace_eval.AgentTrajectoryAccuracy.label"
            for name, pred, cols in [
                ("Relevance", eval_df["eval.Relevance.label"].notna(), ["context.span_id", "eval.Relevance.label", "eval.Relevance.score", "eval.Relevance.explanation"]),
                ("Hallucination", eval_df["eval.Hallucination.label"].notna(), ["context.span_id", "eval.Hallucination.label", "eval.Hallucination.score", "eval.Hallucination.explanation"]),
                ("AgentTrajectory", eval_df[agent_col].notna(), ["context.span_id", agent_col, agent_col.replace(".label", ".score"), agent_col.replace(".label", ".explanation")]),
            ]:
                df = eval_df.loc[pred, cols].copy()
                if len(df) == 0:
                    continue
                for start in range(0, len(df), chunk_size):
                    chunk = df.iloc[start : start + chunk_size]
                    client.log_evaluations_sync(chunk, project_name)
                    total_logged += len(chunk)
            session_cols = ["context.span_id", "session_eval.SearchAugmentSession.label", "session_eval.SearchAugmentSession.score", "session_eval.SearchAugmentSession.explanation"]
            if "session_eval.SearchAugmentSession.label" in eval_df.columns:
                session_df = eval_df[eval_df["session_eval.SearchAugmentSession.label"].notna()][session_cols].copy()
                for start in range(0, len(session_df), chunk_size):
                    chunk = session_df.iloc[start : start + chunk_size]
                    client.log_evaluations_sync(chunk, project_name)
                    total_logged += len(chunk)
            print(f"Successfully logged {total_logged:,} evaluations to Arize. Check the dashboard to verify.")
        except Exception as e:
            print(f"Error logging evaluations: {e}. eval_df is available for manual export.")
    else:
        print("No evaluations generated. Check span data collection.")

Generating evaluations from collected span data...

Collected 2,500 spans
  - CHAIN: 500
  - RETRIEVER: 500
  - RERANKER: 500
  - AGENT: 500
  - LLM: 500

Generating evaluations for AGENT, LLM spans...

Generating session evaluations from CHAIN spans...
  Processing 150 unique sessions for session evaluations
  Added 150 session evaluations

Generated 1,650 evaluations

  - AgentTrajectoryAccuracy: 500
  - Hallucination: 500
  - Relevance: 500
  - Session evaluations: 150
    AgentTrajectoryAccuracy: 356 pass, 144 fail
    Hallucination: 429 pass, 71 fail
    Relevance: 421 pass, 79 fail
    Session evaluation: 116 pass, 34 fail

Logging evaluations to Arize...

  arize.utils.logging | ERROR | Error logging evaluation data to Arize
Traceback (most recent call last):
  File "/Users/camyoung/Documents/Projects/venv_adk_a2a/lib/python3.10/site-packages/arize/pandas/logger.py", line 1810, in _log_arrow_flight
    with flight_writer:
  File "pyarrow/ipc.pxi", line 630, in pyarrow.lib._CReco

In [ ]:
# Synthetic data config: user queries, catalog, first-pull, reranked, personalized responses
import random

USER_QUERIES = [
    "something light and funny for Friday night",
    "thriller with a twist ending",
    "documentary about space exploration",
    "feel-good romance for a rainy day",
    "mind-bending sci-fi like Inception",
    "historical drama set in the 1920s",
    "animated family movie everyone will enjoy",
    "true crime documentary series",
]

# Synthetic catalog: id, title, genre, short description
CATALOG = [
    {"id": "mov_001", "title": "Cosmic Horizons", "genre": "documentary", "description": "A journey through the solar system and beyond."},
    {"id": "mov_002", "title": "Friday Night Laughs", "genre": "comedy", "description": "A group of friends navigate a chaotic evening."},
    {"id": "mov_003", "title": "The Twisted Truth", "genre": "thriller", "description": "Nothing is as it seems in this twist-filled mystery."},
    {"id": "mov_004", "title": "Rainy Day Hearts", "genre": "romance", "description": "Two strangers find love during a storm."},
    {"id": "mov_005", "title": "Recursion", "genre": "sci-fi", "description": "Dreams within dreams; reality bends."},
    {"id": "mov_006", "title": "Jazz Age", "genre": "historical", "description": "1920s New York through the eyes of a musician."},
    {"id": "mov_007", "title": "Skyward", "genre": "animated", "description": "A young bird discovers the meaning of family."},
    {"id": "mov_008", "title": "The Heist Files", "genre": "documentary", "description": "The untold story of the century's biggest heist."},
    {"id": "mov_009", "title": "Last Laugh", "genre": "comedy", "description": "A stand-up comic's road to redemption."},
    {"id": "mov_010", "title": "Double Blind", "genre": "thriller", "description": "A detective uncovers a conspiracy with a shocking end."},
]

# Per-query (or category): first-pull = list of (catalog_item, score); reranked = reordered + optional score tweaks
# Map query index or category to (first_pull_ids_with_scores, reranked_ids_with_scores, response_text)
def _doc_entry(item_id, score):
    item = next((c for c in CATALOG if c["id"] == item_id), None)
    if not item:
        return {"id": item_id, "content": "", "score": score}
    content = f"{item['title']} ({item['genre']}): {item['description']}"
    return {"id": item_id, "content": content, "score": score}

SEARCH_AUGMENT_SCENARIOS = [
    # query_idx 0: light and funny
    {
        "query": USER_QUERIES[0],
        "first_pull": [("mov_002", 0.92), ("mov_009", 0.88), ("mov_007", 0.82), ("mov_004", 0.78), ("mov_001", 0.65)],
        "reranked": [("mov_002", 0.96), ("mov_009", 0.94), ("mov_007", 0.90)],
        "response": "For Friday night I'd go with **Friday Night Laughs** — pure chaos and laughs. If you want something the whole family can enjoy, **Skyward** is sweet and funny. **Last Laugh** is another solid pick if you love stand-up.",
    },
    # query_idx 1: thriller with twist
    {
        "query": USER_QUERIES[1],
        "first_pull": [("mov_003", 0.94), ("mov_010", 0.91), ("mov_005", 0.85), ("mov_006", 0.72), ("mov_008", 0.68)],
        "reranked": [("mov_003", 0.98), ("mov_010", 0.95), ("mov_005", 0.88)],
        "response": "You'll love **The Twisted Truth** — it's exactly what you asked for, with a twist that pays off. **Double Blind** is another great thriller with a conspiracy angle. For something more mind-bending, **Recursion** has the twist factor in a sci-fi package.",
    },
    # query_idx 2: documentary space
    {
        "query": USER_QUERIES[2],
        "first_pull": [("mov_001", 0.95), ("mov_008", 0.75), ("mov_005", 0.70), ("mov_003", 0.55), ("mov_007", 0.50)],
        "reranked": [("mov_001", 0.99), ("mov_008", 0.85), ("mov_005", 0.72)],
        "response": "**Cosmic Horizons** is the one — a proper space documentary that takes you through the solar system and beyond. If you're also into true crime, **The Heist Files** is a gripping documentary series.",
    },
    # query_idx 3: feel-good romance
    {
        "query": USER_QUERIES[3],
        "first_pull": [("mov_004", 0.93), ("mov_002", 0.85), ("mov_007", 0.80), ("mov_001", 0.62), ("mov_006", 0.60)],
        "reranked": [("mov_004", 0.97), ("mov_002", 0.91), ("mov_007", 0.86)],
        "response": "**Rainy Day Hearts** is perfect for a rainy day — cozy and romantic. **Friday Night Laughs** has a sweet subplot too. For something the whole family can enjoy, **Skyward** has a lot of heart.",
    },
    # query_idx 4: mind-bending sci-fi
    {
        "query": USER_QUERIES[4],
        "first_pull": [("mov_005", 0.94), ("mov_003", 0.88), ("mov_001", 0.79), ("mov_010", 0.76), ("mov_006", 0.65)],
        "reranked": [("mov_005", 0.98), ("mov_003", 0.92), ("mov_001", 0.85)],
        "response": "**Recursion** is the closest to that Inception vibe — dreams within dreams and reality bending. **The Twisted Truth** has a mind-bend of its own. **Cosmic Horizons** gives you the big-picture sci-fi feel.",
    },
    # query_idx 5: historical 1920s
    {
        "query": USER_QUERIES[5],
        "first_pull": [("mov_006", 0.96), ("mov_002", 0.70), ("mov_004", 0.68), ("mov_008", 0.65), ("mov_010", 0.60)],
        "reranked": [("mov_006", 0.99), ("mov_004", 0.82), ("mov_008", 0.78)],
        "response": "**Jazz Age** is right in the 1920s — New York, music, and that era's energy. **Rainy Day Hearts** has a period feel in places. **The Heist Files** documentary touches on that era too.",
    },
    # query_idx 6: animated family
    {
        "query": USER_QUERIES[6],
        "first_pull": [("mov_007", 0.95), ("mov_002", 0.88), ("mov_009", 0.85), ("mov_004", 0.75), ("mov_001", 0.65)],
        "reranked": [("mov_007", 0.98), ("mov_002", 0.92), ("mov_009", 0.89)],
        "response": "**Skyward** is the one — animated, family-friendly, and full of heart. **Friday Night Laughs** and **Last Laugh** are great if you want more comedy with the family.",
    },
    # query_idx 7: true crime documentary
    {
        "query": USER_QUERIES[7],
        "first_pull": [("mov_008", 0.94), ("mov_001", 0.78), ("mov_003", 0.72), ("mov_010", 0.70), ("mov_006", 0.65)],
        "reranked": [("mov_008", 0.97), ("mov_010", 0.88), ("mov_003", 0.85)],
        "response": "**The Heist Files** is the true crime documentary series you want — the story of the century's biggest heist. **Double Blind** has a documentary-style tension, and **The Twisted Truth** has that investigative angle.",
    },
]

# Synthetic user profiles / watch history (for fetch_user_history step and hallucination eval)
USER_PROFILES = [
    {"user_id": "u1", "recent_watches": "Cosmic Horizons, Friday Night Laughs", "preferences": "documentaries, comedy", "last_active": "2 days ago"},
    {"user_id": "u2", "recent_watches": "The Twisted Truth, Dark Signal", "preferences": "thrillers, mystery", "last_active": "1 day ago"},
    {"user_id": "u3", "recent_watches": "Rainy Day Hearts, Skyward", "preferences": "romance, family", "last_active": "3 hours ago"},
    {"user_id": "u4", "recent_watches": "Recursion, The Twisted Truth", "preferences": "sci-fi, psychological", "last_active": "5 hours ago"},
    {"user_id": "u5", "recent_watches": "The Heist Files, Beyond Earth", "preferences": "documentaries, true crime", "last_active": "1 day ago"},
]

def _format_user_history(profile: dict) -> str:
    return (
        f"User {profile['user_id']}: recent watches — {profile['recent_watches']}; "
        f"preferences — {profile['preferences']}; last active — {profile['last_active']}"
    )

def get_search_augment_data(query_id=None):
    """
    Get synthetic data for one search-augment trace.
    query_id: int index into SEARCH_AUGMENT_SCENARIOS, or None for random.
    Returns: dict with query, retrieved_docs (first-pull), reranked_docs, response (personalized text).
    """
    if query_id is None:
        query_id = random.randint(0, len(SEARCH_AUGMENT_SCENARIOS) - 1)
    scenario = SEARCH_AUGMENT_SCENARIOS[query_id]
    query = scenario["query"]
    first_pull = [_doc_entry(uid, s) for uid, s in scenario["first_pull"]]
    reranked = [_doc_entry(uid, s) for uid, s in scenario["reranked"]]
    response = scenario["response"]
    profile = random.choice(USER_PROFILES)
    user_history = _format_user_history(profile)
    return {
        "query_id": query_id,
        "query": query,
        "retrieved_docs": first_pull,
        "reranked_docs": reranked,
        "response": response,
        "user_history": user_history,
        "user_profile": profile,
    }

In [ ]:
# Prompt template for LLM: given user query, user history (profile), and reranked recommendations
PROMPT_TEMPLATE = (
    "Given the user's request, their profile/watch history, and the reranked recommendations below, "
    "produce a short conversational response recommending 2-3 titles. Use only titles from the list; "
    "do not invent or fabricate details from the user's profile or history.\n\n"
    "User request: {query}\n\n"
    "User profile / watch history:\n{user_history}\n\n"
    "Reranked recommendations:\n{reranked_list}\n\n"
    "Respond with your personalized picks."
)

def format_llm_input(query: str, reranked_docs: list, user_history: str = "") -> str:
    """Format the LLM input from query, user history, and reranked doc list."""
    lines = [f"- {d['id']}: {d['content']}" for d in reranked_docs]
    return PROMPT_TEMPLATE.format(
        query=query,
        user_history=user_history or "(no profile provided)",
        reranked_list="\n".join(lines),
    )

In [ ]:
# Trace function: CHAIN -> RETRIEVER -> RERANKER -> AGENT -> LLM (all synthetic, no real APIs)
from opentelemetry import trace
from opentelemetry.trace.status import Status, StatusCode
import time
import random

def create_search_augment_trace(tracer, query_id=None, session_id=None):
    """
    Create a single StreamFlix search-augment trace:
    CHAIN -> RETRIEVER (candidate_retrieval) -> RERANKER (rerank_candidates) -> AGENT (search_augment_agent) -> LLM (personalize_recommendations).
    Returns span_data dict: chain_span_id, retrieval_span_id, reranker_span_id, agent_span_id, llm_span_id, query, response, session_id, etc.
    """
    data = get_search_augment_data(query_id)
    query = data["query"]
    retrieved_docs = data["retrieved_docs"]
    reranked_docs = data["reranked_docs"]
    response = data["response"]
    user_history = data.get("user_history", "")
    user_profile = data.get("user_profile", {})
    llm_input = format_llm_input(query, reranked_docs, user_history)

    chain_name = "StreamFlix Search Augment Pipeline"
    retrieval_latency_ms = random.uniform(80, 300)
    reranker_latency_ms = random.uniform(50, 200)
    agent_latency_ms = random.uniform(30, 150)
    user_history_latency_ms = random.uniform(30, 120)
    llm_latency_ms = random.uniform(1200, 3500)
    chain_overhead_ms = random.uniform(20, 80)
    chain_latency_ms = retrieval_latency_ms + reranker_latency_ms + agent_latency_ms + user_history_latency_ms + llm_latency_ms + chain_overhead_ms

    prompt_tokens, completion_tokens = 450, 120
    chain_span_id = retrieval_span_id = reranker_span_id = agent_span_id = user_history_span_id = llm_span_id = None

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", query)
        chain_span.set_attribute("input.mime_type", "text/plain")
        if session_id:
            chain_span.set_attribute("session.id", session_id)
        time.sleep(0.003)
        ctx = chain_span.get_span_context()
        chain_span_id = format(ctx.span_id, "016x")

        with tracer.start_as_current_span("candidate_retrieval") as ret_span:
            ret_span.set_attribute("openinference.span.kind", "RETRIEVER")
            ret_span.set_attribute("input.value", json.dumps({"query": query}))
            ret_span.set_attribute("input.mime_type", "application/json")
            for idx, doc in enumerate(retrieved_docs):
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.id", doc["id"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
            ret_span.set_attribute("output.value", json.dumps([{"id": d["id"], "score": d["score"]} for d in retrieved_docs]))
            ret_span.set_attribute("output.mime_type", "application/json")
            ret_span.set_attribute("latency_ms", round(retrieval_latency_ms, 2))
            ret_span.set_status(Status(StatusCode.OK))
            time.sleep(0.002)
            retrieval_span_id = format(ret_span.get_span_context().span_id, "016x")

        # Human-readable reranker input/output so Arize renders them (it mainly shows input.value / output.value)
        def _doc_label(d):
            c = d.get("content", "")
            return c.split(":")[0].strip() if ":" in c else (c[:50] + "..." if len(c) > 50 else c)
        reranker_input_lines = [
            f"User query: {query}",
            "",
            "Candidates from retrieval (id, title, score):",
        ] + [f"  • {d['id']}: {_doc_label(d)} — {d['score']}" for d in retrieved_docs]
        reranker_input_value = "\n".join(reranker_input_lines)
        reranker_output_lines = [
            "Reranked top picks (passed to LLM for personalization):",
        ] + [f"  {i+1}. {_doc_label(d)} — score {d['score']}" for i, d in enumerate(reranked_docs)]
        reranker_output_value = "\n".join(reranker_output_lines)

        with tracer.start_as_current_span("rerank_candidates") as rerank_span:
            rerank_span.set_attribute("openinference.span.kind", "RERANKER")
            rerank_span.set_attribute("input.value", reranker_input_value)
            rerank_span.set_attribute("input.mime_type", "text/plain")
            rerank_span.set_attribute("output.value", reranker_output_value)
            rerank_span.set_attribute("output.mime_type", "text/plain")
            rerank_span.set_attribute("reranker.model_name", "synthetic-reranker-v1")
            rerank_span.set_attribute("latency_ms", round(reranker_latency_ms, 2))
            rerank_span.set_status(Status(StatusCode.OK))
            time.sleep(0.002)
            reranker_span_id = format(rerank_span.get_span_context().span_id, "016x")

        with tracer.start_as_current_span("search_augment_agent") as agent_span:
            agent_span.set_attribute("openinference.span.kind", "AGENT")
            agent_span.set_attribute("agent.name", "search_augment_agent")
            agent_input = json.dumps({"request": "personalize_recommendations", "query": query, "task": "retrieve -> rerank -> personalize"})
            agent_span.set_attribute("input.value", agent_input)
            agent_span.set_attribute("input.mime_type", "application/json")
            if session_id:
                agent_span.set_attribute("session.id", session_id)
            agent_span.set_attribute("latency_ms", round(agent_latency_ms, 2))
            time.sleep(0.002)
            agent_span_id = format(agent_span.get_span_context().span_id, "016x")

            # Child of agent: fetch user history (profile / watch history) for personalization
            with tracer.start_as_current_span("fetch_user_history") as user_hist_span:
                user_hist_span.set_attribute("openinference.span.kind", "RETRIEVER")
                user_hist_span.set_attribute("input.value", json.dumps({"session_id": session_id, "user_id": user_profile.get("user_id", "unknown")}))
                user_hist_span.set_attribute("input.mime_type", "application/json")
                user_hist_span.set_attribute("output.value", user_history)
                user_hist_span.set_attribute("output.mime_type", "text/plain")
                user_hist_span.set_attribute("latency_ms", round(user_history_latency_ms, 2))
                user_hist_span.set_status(Status(StatusCode.OK))
                time.sleep(0.001)
                user_history_span_id = format(user_hist_span.get_span_context().span_id, "016x")

            with tracer.start_as_current_span("personalize_recommendations") as llm_span:
                if session_id:
                    llm_span.set_attribute("session.id", session_id)
                llm_span.set_attribute("openinference.span.kind", "LLM")
                llm_span.set_attribute("llm.model_name", BEDROCK_MODEL_ID)
                llm_span.set_attribute("llm.provider", "aws")
                llm_span.set_attribute("llm.input_messages.0.message.role", "user")
                llm_span.set_attribute("llm.input_messages.0.message.content", llm_input[:4000])
                llm_span.set_attribute("llm.output_messages.0.message.role", "assistant")
                llm_span.set_attribute("llm.output_messages.0.message.content", response)
                llm_span.set_attribute("llm.token_count.prompt", prompt_tokens)
                llm_span.set_attribute("llm.token_count.completion", completion_tokens)
                llm_span.set_attribute("llm.token_count.total", prompt_tokens + completion_tokens)
                llm_span.set_attribute("input.value", llm_input[:2000])
                llm_span.set_attribute("output.value", response)
                llm_span.set_attribute("latency_ms", round(llm_latency_ms, 2))
                llm_span.set_status(Status(StatusCode.OK))
                llm_span_id = format(llm_span.get_span_context().span_id, "016x")

            agent_span.set_attribute("output.value", response)
            agent_span.set_attribute("output.mime_type", "text/plain")

        chain_span.set_attribute("output.value", response[:500])
        chain_span.set_attribute("output.mime_type", "text/plain")
        chain_span.set_attribute("latency_ms", round(chain_latency_ms, 2))
        chain_span.set_status(Status(StatusCode.OK))

    return {
        "span_data": {
            "chain_span_id": chain_span_id,
            "retrieval_span_id": retrieval_span_id,
            "reranker_span_id": reranker_span_id,
            "agent_span_id": agent_span_id,
            "user_history_span_id": user_history_span_id,
            "llm_span_id": llm_span_id,
            "query": query,
            "response": response,
            "user_history": user_history,
            "query_id": data["query_id"],
            "session_id": session_id,
        }
    }

In [19]:
# Session-level evaluation: attach to root span of first trace per session (Arize session evals)
# https://arize.com/docs/ax/evaluate/evaluators/trace-and-session-evals/session-level-evaluations#5-log-the-results-back-to-arize
def generate_session_evaluations_from_chain_spans(span_df: pd.DataFrame) -> list:
    """
    Generate session-level evaluations. Groups by session_id, gets the root span of the
    first trace in each session (oldest CHAIN span; CHAIN is root in our pipeline).
    Uses session_eval.SearchAugmentSession.* so Arize displays them as session evals.
    """
    session_evaluations = []
    chain_spans = span_df[span_df["openinference.span.kind"] == "CHAIN"].copy()
    if len(chain_spans) == 0:
        return session_evaluations
    if "session_id" not in chain_spans.columns:
        return session_evaluations
    if "timestamp" not in chain_spans.columns:
        chain_spans["timestamp"] = chain_spans.index
    chain_spans = chain_spans[chain_spans["session_id"].notna()]
    if len(chain_spans) == 0:
        return session_evaluations
    chain_spans_sorted = chain_spans.sort_values("timestamp")
    first_trace_root_per_session = chain_spans_sorted.groupby("session_id").first().reset_index()
    for _, row in first_trace_root_per_session.iterrows():
        span_id = row.get("context.span_id", "")
        session_id = row.get("session_id", "unknown")
        query = row.get("query", "")
        response = row.get("response", "")
        will_pass = random.random() < 0.85
        if will_pass:
            label, score = "pass", 0
            explanations = [
                f"Search augment pipeline passed: Session {session_id} produced a coherent personalized response for query '{query[:50]}...'.",
                f"Session validation successful: The end-to-end search augment pipeline for session {session_id} completed correctly.",
            ]
        else:
            label, score = "fail", 1
            explanations = [
                f"Search augment pipeline failed: Session {session_id} had issues in retrieval, rerank, or personalization.",
                f"Session validation failed: The search augment pipeline for session {session_id} did not complete effectively.",
            ]
        session_evaluations.append({
            "context.span_id": span_id,
            "session_eval.SearchAugmentSession.label": label,
            "session_eval.SearchAugmentSession.score": score,
            "session_eval.SearchAugmentSession.explanation": random.choice(explanations),
        })
    return session_evaluations

In [20]:
# Evaluation generation: AGENT (trajectory/orchestration), LLM (relevance, hallucination). CHAIN/RETRIEVER/RERANKER skipped for per-span.
def generate_agent_trajectory_eval(span_data: dict) -> dict:
    """Agent orchestration: did search_augment_agent sequence retrieval -> rerank -> LLM correctly?"""
    will_pass = random.random() < 0.82
    if will_pass:
        label, score = "pass", 0
        explanations = [
            "Search augment orchestration passed: The agent correctly sequenced candidate_retrieval, rerank_candidates, and personalize_recommendations.",
            "Agent trajectory validation successful: The agent followed the expected retrieval -> rerank -> LLM flow.",
        ]
    else:
        label, score = "fail", 1
        explanations = [
            "Search augment orchestration failed: The agent deviated from the expected pipeline sequence.",
            "Agent trajectory validation failed: The agent made suboptimal orchestration decisions.",
        ]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_relevance_eval(span_data: dict) -> dict:
    """Relevance: does the LLM response match the user query and recommendations?"""
    will_pass = random.random() < 0.88
    if will_pass:
        label, score = "pass", 0
        explanations = ["Relevance passed: The personalized response aligns with the user request and recommended titles."]
    else:
        label, score = "fail", 1
        explanations = ["Relevance failed: The response does not adequately address the user query or recommendations."]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_hallucination_eval(span_data: dict) -> dict:
    """Hallucination: does the LLM response stay factual (no fabricated titles or user profile/history)?"""
    will_pass = random.random() < 0.90
    if will_pass:
        label, score = "pass", 0
        explanations = [
            "Hallucination check passed: The response is consistent with the reranked catalog and does not fabricate content or user history.",
        ]
    else:
        label, score = "fail", 1
        explanations = [
            "Hallucination detected: The LLM response made up or invented details from the user's profile or watch history that were not in the retrieved user history.",
            "Hallucination detected: The response fabricated user history (e.g. false recent watches or preferences) not present in the user profile.",
        ]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def create_evaluations_from_span_data(span_df: pd.DataFrame) -> pd.DataFrame:
    """Create evaluation DataFrame. Handles CHAIN (skip), RETRIEVER/RERANKER (skip), AGENT, LLM."""
    evaluations = []
    for _, row in span_df.iterrows():
        span_kind = row.get("openinference.span.kind", "")
        span_id = row.get("context.span_id", "")
        span_data = {
            "query": row.get("query", ""),
            "response": row.get("response", ""),
            "query_id": row.get("query_id", 0),
        }
        if span_kind == "CHAIN":
            continue
        if span_kind in ("RETRIEVER", "RERANKER"):
            continue
        if span_kind == "AGENT":
            eval_data = generate_agent_trajectory_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "trace_eval.AgentTrajectory.label": eval_data["label"],
                "trace_eval.AgentTrajectory.score": eval_data["score"],
                "trace_eval.AgentTrajectory.explanation": eval_data["explanation"],
            })
        elif span_kind == "LLM":
            rel = generate_relevance_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "eval.Relevance.label": rel["label"],
                "eval.Relevance.score": rel["score"],
                "eval.Relevance.explanation": rel["explanation"],
            })
            hal = generate_hallucination_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "eval.Hallucination.label": hal["label"],
                "eval.Hallucination.score": hal["score"],
                "eval.Hallucination.explanation": hal["explanation"],
            })
    return pd.DataFrame(evaluations)

# Single-trace test
Run `create_search_augment_trace` once and flush so one trace appears in Arize.

In [21]:
from opentelemetry import trace

tracer = trace.get_tracer(__name__)
result = create_search_augment_trace(tracer=tracer, query_id=0, session_id="test_session_001")
sd = result["span_data"]
print(f"Created search-augment trace for query: {sd['query'][:60]}...")
print(f"  chain_span_id: {sd['chain_span_id']}, retrieval: {sd['retrieval_span_id']}, reranker: {sd['reranker_span_id']}")
print(f"  agent_span_id: {sd['agent_span_id']}, llm_span_id: {sd['llm_span_id']}")
print(f"  response: {sd['response'][:120]}...")
flush_success = trace.get_tracer_provider().force_flush(timeout_millis=30000)
print("✅ Single trace exported successfully" if flush_success else "⚠️ Flush timeout - span may not have been exported")

Created search-augment trace for query: something light and funny for Friday night...
  chain_span_id: e25c7590013d04d8, retrieval: 7add454825ef0d68, reranker: 046338c33f505e42
  agent_span_id: b9530706a5d4ae3d, llm_span_id: 4c222b79f6932fc7
  response: For Friday night I'd go with **Friday Night Laughs** — pure chaos and laughs. If you want something the whole family can...
✅ Single trace exported successfully


# Batch generation
Generate N traces (e.g. 500), random query_id and optional session_id; collect span_data for CHAIN, RETRIEVER, RERANKER, AGENT, LLM. Flush after batch.

In [22]:
from opentelemetry import trace

tracer = trace.get_tracer(__name__)
NUM_TRACES = 500
SESSION_SIZE_MIN, SESSION_SIZE_MAX = 2, 5
print(f"Generating {NUM_TRACES:,} search-augment traces (CHAIN -> RETRIEVER -> RERANKER -> AGENT -> LLM)...")

span_data_list = []
current_session_id = None
traces_in_session = 0
session_size = 0

for i in range(NUM_TRACES):
    if current_session_id is None or traces_in_session >= session_size:
        current_session_id = f"session_{random.randint(100000, 999999)}"
        session_size = random.randint(SESSION_SIZE_MIN, SESSION_SIZE_MAX)
        traces_in_session = 0
    query_id = random.randint(0, len(SEARCH_AUGMENT_SCENARIOS) - 1)
    result = create_search_augment_trace(tracer=tracer, query_id=query_id, session_id=current_session_id)
    traces_in_session += 1
    if "span_data" not in result:
        continue
    sd = result["span_data"]
    # CHAIN
    span_data_list.append({
        "context.span_id": sd["chain_span_id"],
        "openinference.span.kind": "CHAIN",
        "query_id": sd["query_id"],
        "query": sd["query"],
        "response": sd["response"],
        "session_id": sd.get("session_id"),
    })
    # RETRIEVER
    span_data_list.append({
        "context.span_id": sd["retrieval_span_id"],
        "openinference.span.kind": "RETRIEVER",
        "query_id": sd["query_id"],
        "query": sd["query"],
        "response": sd["response"],
    })
    # RERANKER
    span_data_list.append({
        "context.span_id": sd["reranker_span_id"],
        "openinference.span.kind": "RERANKER",
        "query_id": sd["query_id"],
        "query": sd["query"],
        "response": sd["response"],
    })
    # AGENT
    span_data_list.append({
        "context.span_id": sd["agent_span_id"],
        "openinference.span.kind": "AGENT",
        "query_id": sd["query_id"],
        "query": sd["query"],
        "response": sd["response"],
    })
    # LLM
    span_data_list.append({
        "context.span_id": sd["llm_span_id"],
        "openinference.span.kind": "LLM",
        "query_id": sd["query_id"],
        "query": sd["query"],
        "response": sd["response"],
        "user_history": sd.get("user_history", ""),
    })

print(f"Created {NUM_TRACES:,} traces. Flushing...")
flush_timeout = min(30000 + (NUM_TRACES // 10000) * 10000, 300000)
flush_success = trace.get_tracer_provider().force_flush(timeout_millis=flush_timeout)
print(f"✅ Flush successful. Collected {len(span_data_list):,} span records." if flush_success else "⚠️ Flush timeout.")
collected_span_data = span_data_list

Generating 500 search-augment traces (CHAIN -> RETRIEVER -> RERANKER -> AGENT -> LLM)...
Created 500 traces. Flushing...
✅ Flush successful. Collected 2,500 span records.


# Evaluation generation and logging
Build span DataFrame from collected span data; run `create_evaluations_from_span_data` and session evals; concat and log to Arize via `client.log_evaluations_sync(eval_df, project_name)`.

In [23]:
print(f"{'='*60}")
print("Generating evaluations from collected span data...")
print(f"{'='*60}\n")

if "collected_span_data" not in globals() or not collected_span_data:
    print("⚠️ No span data found. Run the batch generation cell first.")
else:
    span_df = pd.DataFrame(collected_span_data)
    print(f"Collected {len(span_df):,} spans")
    for kind in ["CHAIN", "RETRIEVER", "RERANKER", "AGENT", "LLM"]:
        n = (span_df["openinference.span.kind"] == kind).sum()
        print(f"  - {kind}: {n:,}")
    print("\nGenerating evaluations for AGENT, LLM spans...")
    eval_df = create_evaluations_from_span_data(span_df)
    print("Generating session evaluations from CHAIN spans...")
    session_evaluations = generate_session_evaluations_from_chain_spans(span_df)
    if session_evaluations:
        session_eval_df = pd.DataFrame(session_evaluations)
        eval_df = pd.concat([eval_df, session_eval_df], ignore_index=True)
        print(f"  Added {len(session_evaluations):,} session evaluations")
    if len(eval_df) > 0:
        print(f"\n✅ Generated {len(eval_df):,} evaluations")
        try:
            from arize.pandas.logger import Client
            client = Client(space_id=space_id, api_key=api_key)
            chunk_size = 500
            total_logged = 0
            # Log by eval type: only send rows that have that eval (no n/a placeholders)
            agent_col = "trace_eval.AgentTrajectory.label" if "trace_eval.AgentTrajectory.label" in eval_df.columns else "trace_eval.AgentTrajectoryAccuracy.label"
            for name, pred, cols in [
                ("Relevance", eval_df["eval.Relevance.label"].notna(), ["context.span_id", "eval.Relevance.label", "eval.Relevance.score", "eval.Relevance.explanation"]),
                ("Hallucination", eval_df["eval.Hallucination.label"].notna(), ["context.span_id", "eval.Hallucination.label", "eval.Hallucination.score", "eval.Hallucination.explanation"]),
                ("AgentTrajectory", eval_df[agent_col].notna(), ["context.span_id", agent_col, agent_col.replace(".label", ".score"), agent_col.replace(".label", ".explanation")]),
            ]:
                df = eval_df.loc[pred, cols].copy()
                if len(df) == 0:
                    continue
                for start in range(0, len(df), chunk_size):
                    chunk = df.iloc[start : start + chunk_size]
                    client.log_evaluations_sync(chunk, project_name)
                    total_logged += len(chunk)
            session_cols = ["context.span_id", "session_eval.SearchAugmentSession.label", "session_eval.SearchAugmentSession.score", "session_eval.SearchAugmentSession.explanation"]
            if "session_eval.SearchAugmentSession.label" in eval_df.columns:
                session_df = eval_df[eval_df["session_eval.SearchAugmentSession.label"].notna()][session_cols].copy()
                for start in range(0, len(session_df), chunk_size):
                    chunk = session_df.iloc[start : start + chunk_size]
                    client.log_evaluations_sync(chunk, project_name)
                    total_logged += len(chunk)
            print(f"✅ Logged {total_logged:,} evaluations to Arize")
        except Exception as e:
            print(f"⚠️ Error logging evaluations: {e}")
            print("   eval_df is available for manual log.")
    else:
        print("⚠️ No evaluations generated.")

Generating evaluations from collected span data...

Collected 2,500 spans
  - CHAIN: 500
  - RETRIEVER: 500
  - RERANKER: 500
  - AGENT: 500
  - LLM: 500

Generating evaluations for AGENT, LLM spans...
Generating session evaluations from CHAIN spans...
  Added 147 session evaluations

✅ Generated 1,647 evaluations
  arize.utils.logging | WARNING | ⚠️ Only 441 out of 500 evaluation data were logged, and 59 of data were not logged successfully for model 'streamflix_search_augment'.
  arize.utils.logging | WARNING | ⚠️ Only 441 out of 500 evaluation data were logged, and 59 of data were not logged successfully for model 'streamflix_search_augment'.
  arize.utils.logging | WARNING | ⚠️ Only 435 out of 500 evaluation data were logged, and 65 of data were not logged successfully for model 'streamflix_search_augment'.
  arize.utils.logging | WARNING | ⚠️ Only 131 out of 147 evaluation data were logged, and 16 of data were not logged successfully for model 'streamflix_search_augment'.
✅ Logged